# Tasks 2-4 Walkthrough

Run each cell top-to-bottom. Every component is imported from the project scripts and run on an **input query** so you can see the intermediate output of each step.

- **Task 2A** Decision Attribution  — why is an assignment in the optimal solution?
- **Task 2B** Counterfactual  — what if I change an assignment?
- **Task 2C** Feedback Localization  — vague complaint → structured critique (+ clarify gate)
- **Task 4** TPO refinement loop + end-to-end pipeline trace
- **Task 3** Ablation + Task 2 evaluation metrics

> Task 5 is a written design doc: `docs/task5_system_design.md` (no runtime).

## Setup — point Python at the project root

In [1]:
import os, sys, json
# make the project root the working dir + import path (notebook lives in notebooks/)
ROOT = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(ROOT); sys.path.insert(0, ROOT)
from llm import client
print('project root :', ROOT)
print('LLM active   :', client.available(), '(False = rule/template fallback, still runs)')

project root : c:\Users\tanwe\OneDrive\Desktop\alphaz-technical
LLM active   : True (False = rule/template fallback, still runs)


## Solve the nurse schedule (the source of truth)
CP-SAT solves to optimality. Everything downstream explains or refines THIS solution.

In [2]:
from solver.solve import solve, solution_json
base = solve()
print('feasible :', base['feasible'], '| objective :', base['objective_value'],
      '| shifts :', len(base['assignments']))
# show the schedule as worker -> list of (day,shift)
from collections import defaultdict
sched = defaultdict(list)
for k in base['assignments']:
    w,d,s = k.split(','); sched[w].append(f'{d}-{s}')
for w in sorted(sched): print(f'  {w}: {sched[w]}')

feasible : True | objective : 39.0 | shifts : 32
  A: ['Tue-M', 'Wed-M', 'Thu-M', 'Fri-M']
  B: ['Mon-N', 'Wed-N', 'Thu-E', 'Fri-N']
  C: ['Mon-M', 'Tue-M', 'Wed-M', 'Thu-M']
  D: ['Mon-N', 'Wed-N', 'Thu-E', 'Fri-N']
  E: ['Mon-E', 'Tue-E', 'Thu-E', 'Fri-E']
  F: ['Mon-E', 'Tue-N', 'Wed-E', 'Thu-N']
  G: ['Mon-E', 'Tue-N', 'Wed-E', 'Fri-M']
  H: ['Mon-M', 'Tue-E', 'Thu-N', 'Fri-E']


## Task 2A — Decision Attribution
**Input:** a specific assignment, e.g. `A,Mon,M`.  
**Output:** which hard constraints force it (IIS), which objective terms prefer it, LP dual values / reduced costs, and the objective change if it were removed.

In [3]:
from explain.attribution import attribute
from llm.nl_explainer import explain_attribution

TARGET = 'A,Mon,M'          # <- change this to any assigned (worker,day,shift)
attr = attribute(TARGET, base_sol=base)
print(json.dumps(attr, indent=2, default=str))
print('\nNatural language:\n', explain_attribution(attr))

{
  "target_assignment": "x[A,Mon,M] = 1",
  "forced_by_constraints": [],
  "preferred_by_objective_terms": [
    "fair_workload_balance"
  ],
  "dual_values": {
    "coverage_Mon_M": 2.0
  },
  "reduced_costs": {
    "x_A_Mon_M": 0.0
  },
  "objective_delta_if_removed": 0.0,
  "method": "IIS(assumptions) + objective-delta + LP-duals"
}

Natural language:
 Here's an explanation of the optimization attribution in simple terms:

This optimization decision assigned a value of 1 to a specific task 'x' on Monday, and it was made because it helps to balance the workload fairly among different tasks. The decision was also influenced by the fact that it doesn't conflict with any other constraints, and it was chosen because it has a positive effect on the overall objective of the system.


## Task 2B — Counterfactual
**Input:** a requested change, e.g. swap A and D on Wed Night.  
**Output:** feasibility, objective change, knock-on changes, latency. The local `counterfactual` (fix-and-reoptimize) is compared against a full re-solve for the Task 2 consistency metric.

In [4]:
from explain.counterfactual import counterfactual, full_resolve

CHANGES = {('A','Wed','N'): 0, ('D','Wed','N'): 1}   # <- edit the what-if here
local = counterfactual(base, CHANGES)
full  = full_resolve(base, CHANGES)
print('LOCAL (fix-and-reoptimize):'); print(json.dumps(local, indent=2, default=str))
print('\nFULL re-solve (ground truth):'); print(json.dumps(full, indent=2, default=str))
print('\nfeasibility agree:', local['feasible']==full['feasible'],
      '| local latency: %.3fs'%local['latency_s'])

LOCAL (fix-and-reoptimize):
{
  "feasible": true,
  "objective_value": 39.0,
  "objective_delta": 0.0,
  "knock_on_changes": [],
  "latency_s": 0.2647,
  "method": "fix-and-reoptimize (local neighborhood)"
}

FULL re-solve (ground truth):
{
  "feasible": true,
  "objective_value": 39.0,
  "knock_on_changes": [
    "C,Fri,M",
    "C,Mon,M",
    "F,Mon,E",
    "F,Mon,M"
  ],
  "latency_s": 0.0916
}

feasibility agree: True | local latency: 0.265s


## Task 2C — Feedback Localization (+ clarify gate)
**Input:** a vague natural-language complaint.  
**Output:** structured critique `{target_id, modification_type, confidence}`. Low confidence → a clarifying question is returned instead of triggering TPO.

In [5]:
from explain.localization import localize

for complaint in ['Worker A is too tired',
                  'This schedule is unfair',
                  'Too many night shifts for Worker C',
                  'huh this looks weird']:
    r = localize(complaint)
    print(f'> {complaint!r}')
    print('   action:', r['action'])
    print('  ', json.dumps(r.get('localization'), default=str))
    if r['action']=='clarify': print('   QUESTION:', r['question'])
    print()

> 'Worker A is too tired'
   action: refine
   {"target_type": "worker_constraint", "target_id": "max_shifts_A", "modification_type": "add_soft_constraint", "critique": "Reduce Worker A's total shifts / avoid demanding back-to-back shifts.", "confidence": 0.85}

> 'This schedule is unfair'
   action: refine
   {"target_type": "objective_term", "target_id": "fair_workload_balance", "modification_type": "adjust_objective_weight", "critique": "Increase the fairness weight to balance workload across workers.", "confidence": 0.75}

> 'Too many night shifts for Worker C'
   action: refine
   {"target_type": "worker_constraint", "target_id": "max_shifts_C", "modification_type": "add_soft_constraint", "critique": "Reduce Worker C's total shifts / avoid demanding back-to-back shifts.", "confidence": 0.85}

> 'huh this looks weird'
   action: clarify
   {"target_type": "unknown", "target_id": null, "modification_type": null, "critique": "huh this looks weird", "confidence": 0.3}
   QUESTION: I'm

## Task 4 — TPO refinement loop
Take the structured critique from localization, inject it as a soft constraint, and re-solve iteratively until the complaint is resolved (or D iterations).

In [6]:
from tpo_refine.refine_loop import refine, _critique_to_soft

critique = localize('Worker A is too tired')['localization']
print('critique  :', json.dumps(critique, default=str))
print('soft constraint:', _critique_to_soft(critique))

refined, traj, converged = refine(critique, base_sol=base)
print('\nrefinement trajectory:')
for it in traj: print('  ', it)
print('\nconverged:', converged,
      '| objective', base['objective_value'], '->', refined['objective_value'])

critique  : {"target_type": "worker_constraint", "target_id": "max_shifts_A", "modification_type": "add_soft_constraint", "critique": "Reduce Worker A's total shifts / avoid demanding back-to-back shifts.", "confidence": 0.85}
soft constraint: {'id': 'soft_reduce_A', 'vars': [('A', 'Mon', 'M'), ('A', 'Mon', 'E'), ('A', 'Mon', 'N'), ('A', 'Tue', 'M'), ('A', 'Tue', 'E'), ('A', 'Tue', 'N'), ('A', 'Wed', 'M'), ('A', 'Wed', 'E'), ('A', 'Wed', 'N'), ('A', 'Thu', 'M'), ('A', 'Thu', 'E'), ('A', 'Thu', 'N'), ('A', 'Fri', 'M'), ('A', 'Fri', 'E'), ('A', 'Fri', 'N')], 'limit': 2, 'weight': 100}

refinement trajectory:
   {'iteration': 1, 'feasible': True, 'objective_value': 45.0, 'complaint_free': True, 'soft_constraints': ['soft_reduce_A'], 'probe': None}

converged: True | objective 39.0 -> 45.0


## Task 4 — End-to-end pipeline trace
The same flow the FastAPI `/trace` endpoint and the web UI return, as one structured object.

In [8]:
from api.main import post_trace, TraceIn, post_solve
sid = post_solve()['solution_id']
trace = post_trace(TraceIn(complaint='Worker A is too tired', solution_id=sid))
for st in trace['stages']:
    print(f"[{st['id']}] {st['title']}  ({st['status']})")
    print('   ', json.dumps(st['detail'], default=str)[:200], '...')
print('\nstopped:', trace['stopped'])

[1] Baseline solve (CP-SAT)  (ok)
    {"objective_value": 39.0, "num_assignments": 32, "feasible": true} ...
[2] Feedback localization  (ok)
    {"action": "refine", "target_id": "max_shifts_A", "target_type": "worker_constraint", "modification_type": "add_soft_constraint", "confidence": 0.85} ...
[3] Structured critique → soft constraint  (ok)
    {"critique": "Reduce Worker A's total shifts / avoid demanding back-to-back shifts.", "soft_constraint": {"id": "soft_reduce_A", "vars": [["A", "Mon", "M"], ["A", "Mon", "E"], ["A", "Mon", "N"], ["A", ...
[4] TPO refinement loop  (ok)
    {"iterations": [{"iteration": 1, "feasible": true, "objective_value": 45.0, "complaint_free": true, "soft_constraints": ["soft_reduce_A"], "probe": null}], "converged": true} ...
[5] Final solution + feasibility check  (ok)
    {"feasible": true, "objective_before": 39.0, "objective_after": 45.0, "converged": true, "assignments": ["A,Tue,M", "A,Wed,M", "B,Mon,N", "B,Wed,N", "B,Thu,E", "B,Fri,N", "C,Tue,M", 

## Task 3 — Ablation study
Conditions A/B/C, each run ≥3x; reports mean & std of convergence rate (feasible + complaint-free within 3 TPO iters).

> Note: the scaffold's 'without' conditions may still converge — making them genuinely weaker is the analytical work of Task 3 (see README).

In [9]:
from eval.ablation import ablate
for label, kw in [('A: with attribution', dict(use_attribution=True)),
                  ('A: without attribution', dict(use_attribution=False)),
                  ('C: with localization', dict(use_localization=True)),
                  ('C: raw complaint', dict(use_localization=False))]:
    print(ablate(label, runs=3, **kw))

{'condition': 'A: with attribution', 'mean': 0.714, 'std': 0.0, 'runs': [0.7142857142857143, 0.7142857142857143, 0.7142857142857143]}
[llm] rate-limited (attempt 1/6); waiting 2s…
[llm] rate-limited (attempt 2/6); waiting 4s…
[llm] rate-limited (attempt 3/6); waiting 8s…
[llm] rate-limited (attempt 1/6); waiting 2s…
[llm] rate-limited (attempt 2/6); waiting 4s…
[llm] rate-limited (attempt 1/6); waiting 2s…
[llm] rate-limited (attempt 2/6); waiting 4s…
[llm] rate-limited (attempt 1/6); waiting 2s…
[llm] rate-limited (attempt 1/6); waiting 2s…
[llm] rate-limited (attempt 2/6); waiting 4s…
[llm] rate-limited (attempt 1/6); waiting 2s…
[llm] rate-limited (attempt 1/6); waiting 2s…
[llm] rate-limited (attempt 2/6); waiting 4s…
[llm] rate-limited (attempt 3/6); waiting 8s…
[llm] rate-limited (attempt 4/6); waiting 16s…
{'condition': 'A: without attribution', 'mean': 0.0, 'std': 0.0, 'runs': [0.0, 0.0, 0.0]}
[llm] rate-limited (attempt 1/6); waiting 2s…
[llm] rate-limited (attempt 2/6); waiti

{"condition": "A: with attribution", "mean": 0.714, "std": 0.0, "runs": [0.7142857142857143, 0.7142857142857143, 0.7142857142857143]}


{"condition": "A: without attribution", "mean": 0.0, "std": 0.0, "runs": [0.0, 0.0, 0.0]}


{"condition": "B: with cf probing", "mean": 0.143, "std": 0.0, "runs": [0.14285714285714285, 0.14285714285714285, 0.14285714285714285]}


{"condition": "B: without cf probing", "mean": 0.714, "std": 0.0, "runs": [0.7142857142857143, 0.7142857142857143, 0.7142857142857143]}


{"condition": "C: with localization", "mean": 0.714, "std": 0.0, "runs": [0.7142857142857143, 0.7142857142857143, 0.7142857142857143]}


{"condition": "C: raw complaint", "mean": 0.0, "std": 0.0, "runs": [0.0, 0.0, 0.0]}

## Task 2 — Evaluation metrics
Localization accuracy by confidence level + counterfactual consistency vs full re-solve.

In [10]:
from eval.evaluate import eval_localization, eval_counterfactual
print('localization :', eval_localization())
print('counterfactual:', eval_counterfactual())

[llm] rate-limited (attempt 1/6); waiting 2s…
[llm] rate-limited (attempt 1/6); waiting 2s…
[llm] rate-limited (attempt 1/6); waiting 2s…
localization : {'high_conf_acc': 0.833, 'high_n': 6, 'low_conf_acc': 0.5, 'low_n': 2}
counterfactual: {'n': 2, 'agreement_rate': 1.0, 'mean_latency_s': 0.0591}
